# Bài tập 4 — Tự viết bộ lọc bằng tổng log
Tự đếm từ, làm trơn alpha = 1, tính tiên nghiệm và tổng log. Chỉ dùng scikit-learn để chia dữ liệu và làm mô hình đối chiếu. Cả hai bộ lọc dùng cùng tập học, tập kiểm tra và từ điển học từ tập học.

Chọn **Run All** để chạy độc lập notebook này. Trên Colab, tải CSV cần dùng vào Files hoặc thư mục `data/`.


In [1]:
from pathlib import Path

# Chạy từ thư mục bài, code/, hoặc thư mục chứa CSV đã tải lên Colab.
thu_muc_hien_tai = Path.cwd().resolve()
thu_muc_du_an = thu_muc_hien_tai
for thu_muc in (thu_muc_hien_tai, *thu_muc_hien_tai.parents):
    if (thu_muc / "data" / "tin_nhan.csv").is_file():
        thu_muc_du_an = thu_muc
        break

def tep_du_lieu(ten):
    for duong_dan in (thu_muc_du_an / "data" / ten, thu_muc_hien_tai / ten):
        if duong_dan.is_file():
            return duong_dan
    raise FileNotFoundError(f"Không tìm thấy {ten}. Hãy giữ thư mục data/ hoặc tải CSV lên Colab.")

thu_muc_hinh = thu_muc_du_an / "figures"
thu_muc_hinh.mkdir(exist_ok=True)


## Mã và kết quả

In [2]:
from collections import Counter
from math import log
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

df = pd.read_csv(tep_du_lieu("tin_nhan.csv"))
cau_train, cau_test, y_train, y_test = train_test_split(
    df["noi_dung"], df["la_rac"], test_size=0.25, random_state=17, stratify=df["la_rac"]
)

def tach_tu(cau):
    return cau.lower().split()

tu_dien = sorted({tu for cau in cau_train for tu in tach_tu(cau)})
tap_tu = set(tu_dien)
V = len(tu_dien)
dem = {0: Counter(), 1: Counter()}
so_tin = Counter(int(nhan) for nhan in y_train)
for cau, nhan in zip(cau_train, y_train):
    dem[int(nhan)].update(tach_tu(cau))
tong = {nhan: sum(dem[nhan].values()) for nhan in [0, 1]}
alpha = 1.0

def diem_log(cau, nhan):
    # Tiên nghiệm dựa trên số tin, không phải số từ.
    diem = log(so_tin[nhan] / len(y_train))
    for tu, so_lan in Counter(tach_tu(cau)).items():
        if tu in tap_tu:  # Bỏ từ ngoài từ điển, giống CountVectorizer.transform.
            p_tu = (dem[nhan][tu] + alpha) / (tong[nhan] + alpha * V)
            diem += so_lan * log(p_tu)
    return diem

def du_doan(cau):
    # Nếu bằng điểm, chọn lớp 0 như thứ tự classes_ của thư viện.
    return 1 if diem_log(cau, 1) > diem_log(cau, 0) else 0

nhan_tu_viet = np.array([du_doan(cau) for cau in cau_test])
acc_tu_viet = sum(nhan_tu_viet == y_test.to_numpy()) / len(y_test)

vec = CountVectorizer(token_pattern=r"(?u)\S+", vocabulary=tu_dien)
X_train = vec.fit_transform(cau_train)
X_test = vec.transform(cau_test)
model = MultinomialNB(alpha=alpha).fit(X_train, y_train)
nhan_thu_vien = model.predict(X_test)
acc_thu_vien = accuracy_score(y_test, nhan_thu_vien)

print(f"Số tin học: {len(cau_train)}; kiểm tra: {len(cau_test)}; số từ: {V}")
print(f"Accuracy tự viết       = {acc_tu_viet:.4f}")
print(f"Accuracy MultinomialNB = {acc_thu_vien:.4f}")
print(f"Số dự đoán khác nhau  = {int((nhan_tu_viet != nhan_thu_vien).sum())}")
print("Giải thích: Tổng log tương đương so sánh tích xác suất vì log là hàm đồng biến; cách tính này tránh tích nhiều số nhỏ bị tràn dưới về 0.")
print("Làm trơn cộng alpha vào số đếm và alpha×V vào tổng số từ, nên từ trong từ điển nhưng chưa gặp ở một lớp vẫn có xác suất dương. Từ hoàn toàn ngoài từ điển bị bỏ qua.")

Số tin học: 225; kiểm tra: 75; số từ: 40
Accuracy tự viết       = 0.9467
Accuracy MultinomialNB = 0.9467
Số dự đoán khác nhau  = 0
Giải thích: Tổng log tương đương so sánh tích xác suất vì log là hàm đồng biến; cách tính này tránh tích nhiều số nhỏ bị tràn dưới về 0.
Làm trơn cộng alpha vào số đếm và alpha×V vào tổng số từ, nên từ trong từ điển nhưng chưa gặp ở một lớp vẫn có xác suất dương. Từ hoàn toàn ngoài từ điển bị bỏ qua.
